In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [4]:
df = pd.read_excel('Berlin_wind_multiclass1.xlsx')

In [5]:
df.head()

,Year,Month,Day,Hour,Minute,Temperature,Clearsky.DHI,Clearsky.DNI,Clearsky.GHI,Cloud.Type,...,GHI,Ozone,Relative.Humidity,Solar.Zenith.Angle,Surface.Albedo,Pressure,Precipitable.Water,Wind.Direction,Wind.Speed,X50Hertz..MW.
0,2018,1,1,0,0,7.9,0,0,0,7,...,0,0.287,93.39,148.97,0.12,993,1.9,212,5.6,713.29
1,2018,1,1,0,15,7.9,0,0,0,8,...,0,0.287,93.39,147.97,0.12,993,1.9,213,5.7,701.46
2,2018,1,1,0,30,7.8,0,0,0,8,...,0,0.286,94.03,146.76,0.12,993,2.0,214,5.8,701.96
3,2018,1,1,0,45,7.7,0,0,0,7,...,0,0.286,91.31,145.35,0.12,993,2.0,214,5.8,733.83
4,2018,1,1,1,0,7.7,0,0,0,7,...,0,0.286,91.31,143.78,0.12,993,2.0,215,5.9,753.35


In [7]:
power = df['X50Hertz..MW.']
boundaries = power.quantile([0.2,0.4,0.6,0.8]).values
conditions = [
    power <= boundaries[0],
    (power > boundaries[0]) & (power <= boundaries[1]),
    (power > boundaries[1]) & (power <= boundaries[2]),
    (power > boundaries[2]) & (power <= boundaries[3]),
    power > boundaries[3]
]
choices = [0, 1, 2, 3, 4]
df['target'] = np.select(conditions,choices)

print(df['target'].value_counts().sort_index())

target
0    14026
1    14006
2    14021
3    14012
4    14015
Name: count, dtype: int64


In [27]:
y = df['target']
X = df.drop(['X50Hertz..MW.','target','Year','Minute'],axis = 1)

In [11]:
def exclude_labels(data, labels):
    return data[~data['target'].isin(labels)]

In [12]:
df01 = exclude_labels(df,[0,1])

In [13]:
df23 = exclude_labels(df,[2,3])

In [14]:
df01.head()

,Year,Month,Day,Hour,Minute,Temperature,Clearsky.DHI,Clearsky.DNI,Clearsky.GHI,Cloud.Type,...,Ozone,Relative.Humidity,Solar.Zenith.Angle,Surface.Albedo,Pressure,Precipitable.Water,Wind.Direction,Wind.Speed,X50Hertz..MW.,target
0,2018,1,1,0,0,7.9,0,0,0,7,...,0.287,93.39,148.97,0.12,993,1.9,212,5.6,713.29,4
1,2018,1,1,0,15,7.9,0,0,0,8,...,0.287,93.39,147.97,0.12,993,1.9,213,5.7,701.46,4
2,2018,1,1,0,30,7.8,0,0,0,8,...,0.286,94.03,146.76,0.12,993,2.0,214,5.8,701.96,4
3,2018,1,1,0,45,7.7,0,0,0,7,...,0.286,91.31,145.35,0.12,993,2.0,214,5.8,733.83,4
4,2018,1,1,1,0,7.7,0,0,0,7,...,0.286,91.31,143.78,0.12,993,2.0,215,5.9,753.35,4


In [23]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,f1_score
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

In [18]:
def train_test(df):
    y = df['target']
    X = df.drop(['X50Hertz..MW.','target','Year','Minute'],axis = 1)
    x_train,x_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
    return [x_train,x_test,y_train,y_test]

In [19]:
ch1 = train_test(df01)
ch2 = train_test(df23)

In [24]:


def train_local_model(X_train, y_train):
    encoder = LabelEncoder()
    y_encoded = encoder.fit_transform(y_train)

    model = XGBClassifier(
        objective="multi:softprob",
        num_class=len(encoder.classes_),
        eval_metric="mlogloss",
        tree_method="hist",
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        random_state=42
    )

    model.fit(X_train, y_encoded)
    return model, encoder

model1, encoder1 = train_local_model(ch1[0], ch1[2])
model2, encoder2 = train_local_model(ch2[0], ch2[2])


pred1 = encoder1.inverse_transform(model1.predict(ch1[1]).astype(int))
pred2 = encoder2.inverse_transform(model2.predict(ch2[1]).astype(int))


In [26]:
print(accuracy_score(ch1[3],pred1))
print(accuracy_score(ch2[3],pred2))

0.870986920332937
0.9192627824019025


In [29]:
pred_total_model1 = model1.predict(X)
pred_total_model2 = model2.predict(X)

In [30]:
print(accuracy_score(y,pred_total_model1))

0.20559360730593607


In [31]:
print(accuracy_score(y,pred_total_model2))

0.38323344748858446


In [38]:
import flwr as fl
import xgboost as xgb
from flwr.client import NumPyClient

print("Flower version:", fl.__version__)
print("XGBoost version:", xgb.__version__)

Flower version: 1.40.0
XGBoost version: 3.4.1


In [39]:
client_data = {
    "client1": {
        "X_train": ch1[0],
        "y_train": ch1[2],
        "X_test": ch1[1],
        "y_test": ch1[3]
    },
    "client2": {
        "X_train": ch2[0],
        "y_train": ch2[2],
        "X_test": ch2[1],
        "y_test": ch2[3]
    }
}

print(client_data["client1"]["X_train"].shape)
print(client_data["client2"]["X_train"].shape)

(33638, 21)
(33637, 21)


In [40]:
from flwr.serverapp.strategy import FedXgbBagging

print("FedXgbBagging is available")

FedXgbBagging is available


In [41]:
import inspect
from flwr.serverapp.strategy import FedXgbBagging

print(inspect.signature(FedXgbBagging))

(fraction_train: float = 1.0, fraction_evaluate: float = 1.0, min_train_nodes: int = 2, min_evaluate_nodes: int = 2, min_available_nodes: int = 2, weighted_by_key: str = 'num-examples', arrayrecord_key: str = 'arrays', configrecord_key: str = 'config', train_metrics_aggr_fn: collections.abc.Callable[[list[flwr.app.message.recorddict.RecordDict], str], flwr.app.message.metricrecord.MetricRecord] | None = None, evaluate_metrics_aggr_fn: collections.abc.Callable[[list[flwr.app.message.recorddict.RecordDict], str], flwr.app.message.metricrecord.MetricRecord] | None = None) -> None
